# CellPLM: Spatial Gene Imputation (Pseudo-6K Benchmark)

CellPLM is a pretrained large language model for single-cell biology.
Here it is fine-tuned on CosMx + scRNA-seq data to impute missing genes
in the pseudo-6K spatial experiment.

**Input**: WTX CosMx h5ad (Ensembl gene IDs); scRNA-seq reference; 6K gene list  
**Output**: `cellplm_half_imp_scrna_log1p_trained.csv` — combined 6K observed + CellPLM-imputed 12K genes

> Note: CellPLM requires Ensembl gene IDs. Gene symbols are converted via mygene.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import torch
from scipy.sparse import csr_matrix
from CellPLM.utils import set_seed
from CellPLM.utils.data import stratified_sample_genes_by_sparsity
from CellPLM.pipeline.imputation import ImputationPipeline

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
PRETRAIN_VERSION = '20231027_85M'
DEVICE           = 'cuda:0'
DATA_DIR         = "/path/to/cosmx/data"
SCRNA_DIR        = "/path/to/scrna/data"
OUT_DIR          = "/path/to/cosmx/count_csv"

In [ ]:
# Load WTX CosMx data (log1p normalized)
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_I0261_I0331_anno.h5ad")

# Convert gene symbols to Ensembl IDs (CellPLM requirement)
def symbol_to_ensembl(gene_list):
    import mygene
    mg = mygene.MyGeneInfo()
    return (mg.querymany(gene_list, scopes='symbol', fields='ensembl.gene',
                         as_dataframe=True, species='human')
              .reset_index().drop_duplicates(subset='query')['ensembl.gene']
              .fillna('0').tolist())

ensembl_ids = symbol_to_ensembl(adata.var.index.tolist())
adata.var['ensembl']   = ensembl_ids
adata.var['gene_name'] = adata.var.index.tolist()

# Keep only genes with valid Ensembl IDs
valid_ensembl = adata.var[adata.var['ensembl'] != '0']['ensembl'].tolist()
adata = adata[:, adata.var.ensembl.isin(valid_ensembl)]
adata.var.index   = adata.var.ensembl.tolist()
adata.obs_names   = adata.obs['fov_cell_id'].tolist()
adata.obs['fov']  = adata.obs['fov'].astype(int)
print(f"Genes after Ensembl filtering: {adata.n_vars:,}")

In [ ]:
# Train/test split by FOV
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs.index):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)

test_adata  = adata[test_cells, :].copy()
train_adata = adata[train_cells, :].copy()
print(f"Train: {len(train_cells):,}  |  Test: {len(test_cells):,}")

In [ ]:
# 6K gene panel (pseudo-6K experiment)
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()
genes_6k_ensembl = symbol_to_ensembl(genes_6k)
genes_6k_ensembl = [g for g in genes_6k_ensembl if g in adata.var.index.tolist()]

# Target genes = genes NOT in 6K panel (what CellPLM needs to impute)
query_genes  = genes_6k_ensembl
target_genes = [g for g in adata.var_names if g not in query_genes]
print(f"Query (6K): {len(query_genes)}  |  Target (12K): {len(target_genes)}")

In [ ]:
# Load scRNA-seq reference for CellPLM training
scrna = sc.read_h5ad(f"{SCRNA_DIR}/uc_atlas_scrna_for_cellplm.h5ad")
scrna.obs['batch'] = scrna.obs['patient'].tolist()

# Downsample scRNA reference for computational efficiency
sc.pp.subsample(scrna, n_obs=20000, random_state=42)

# Concatenate: spatial (train+test) + scRNA reference
train_adata.obs['split'] = 'train'
test_adata.obs['split']  = 'test'
scrna.obs['split']       = 'train'
train_data_concat = ad.concat([train_adata, test_adata, scrna])

In [ ]:
# Filter to batches with sufficient cells (> 100) for CellPLM training
batch_sizes = train_data_concat.obs['batch'].value_counts()
large_batches = batch_sizes[batch_sizes > 100].index.tolist()
train_data_concat = train_data_concat[train_data_concat.obs['batch'].isin(large_batches)]
test_adata_filter = test_adata[test_adata.obs['batch'].isin(large_batches)]

In [ ]:
# ── CellPLM pipeline ─────────────────────────────────────────────────────────
batch_gene_list = {
    batch: target_genes if batch in test_adata.obs['batch'].unique()
           else list(scrna.var_names)
    for batch in list(test_adata.obs['batch'].unique()) + list(scrna.obs['batch'].unique())
}

pipeline_config = {
    'target_genes':    target_genes,
    'pretrain_prefix': PRETRAIN_VERSION,
    'batch_gene_list': batch_gene_list,
}

pipeline = ImputationPipeline(pretrain_prefix=PRETRAIN_VERSION, device=DEVICE)
pipeline.fit(
    train_data_concat,
    pipeline_config,
    split_field='split',
    target_split='test',
    max_epochs=10,
)

In [ ]:
# Predict imputed expression for test cells
pred = pipeline.predict(test_adata, pipeline_config, device=DEVICE)
pred_df = pd.DataFrame(pred.cpu())
pred_df.columns = test_adata.var['gene_name'].tolist()  # convert back to symbols

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────
# Observed 6K gene names (symbols)
query_gene_names = test_adata.var[test_adata.var.index.isin(query_genes)]['gene_name'].tolist()

# Observed counts (6K panel)
ori_query = pd.DataFrame(test_adata[:, test_adata.var.gene_name.isin(query_gene_names)].X)
ori_query.columns = query_gene_names

# Combine: 6K observed + CellPLM-imputed 12K
imputed_cols = [c for c in pred_df.columns if c not in query_gene_names]
combined = pd.concat([ori_query, pred_df[imputed_cols]], axis=1)

combined.to_csv(f"{OUT_DIR}/cellplm_half_imp_scrna_log1p_trained.csv")
print(f"Exported: {combined.shape[0]:,} cells x {combined.shape[1]:,} genes")